In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
tqdm.pandas()

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score
sns

import warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import pandas as pd

pokemon_path = os.path.join(path, '/kaggle/input/q1-ka-ai-2026/Q1_data.csv')
fdt = pd.read_csv(pokemon_path)

print(f"Shape: {fdt.shape}")


In [ ]:
fdt.head()

In [ ]:
fdt.info()

In [ ]:
fdt.describe()

In [ ]:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(fdt, "Delivery_Time")

In [ ]:
fdt = fdt.drop('Order_ID', axis=1)

In [ ]:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(fdt)

In [ ]:
mis= ['Weather','Traffic_Level','Courier_Experience_yrs','Time_of_Day','Delivery_Time']

In [ ]:
fdt = fdt.dropna(subset=mis)


In [ ]:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(fdt)

In [ ]:
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))
    return categorical_cols

label_encoders = encode_categorical_columns(fdt)


In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = fdt.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    fdt[col] = le.fit_transform(fdt[col])
fdt

In [ ]:
fdt['Weather']

In [ ]:
fdt

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(fdt)

In [ ]:
import seaborn as sns
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(fdt, "Delivery_Time")

In [ ]:
X = fdt.drop("Delivery_Time",axis=1)
y = fdt['Delivery_Time']

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error , mean_squared_error

# Use previously generated random data (example: regression data)
X, y = X.copy(), y.copy()

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
avrge =[]
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Model trained!")
    mae = mean_absolute_error(y_test, y_pred)
    avrge.append(mae)
np.mean(avrge)

In [ ]:
coeffs = {}
from sklearn.linear_model import Ridge, Lasso

coeffs['Lasso'] = Lasso(alpha=1.0,  max_iter=10000).coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:

plt.hist(y_pred)
plt.title(f"Target Distribution")
plt.xlabel('')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
!pip install catboost

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
         X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
         y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        # Train the model
         model.fit(X_train, y_train)
         y_pred = model.predict(X_test)

        # Calculate metrics
         scores_f1.append(f1_score(y_test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")
    mse = mean_squared_error(y_test, y_pred)